# PySpark — DataFrame Transformations

Transformations create a **new DataFrame** — they are **lazy** (nothing runs until an action like `.show()` or `.count()` is called).

| Operation | Method | Purpose |
|-----------|--------|---------|
| Choose columns | `select()` | Project specific columns |
| Filter rows | `filter()` / `where()` | Keep rows matching a condition |
| Add / modify column | `withColumn()` | Create or overwrite a column |
| Remove column | `drop()` | Delete one or more columns |
| Rename column | `withColumnRenamed()` | Rename without rewriting all columns |
| Sort rows | `orderBy()` / `sort()` | Ascending or descending |
| Remove duplicates | `distinct()` | Unique rows |
| Limit rows | `limit(n)` | First N rows |

## Setup — Create Sample DataFrame

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, lower, lit, round as spark_round

spark = SparkSession.builder.appName("Transformations").getOrCreate()

data = [
    ("Alice",   "Engineering", 95000, 5),
    ("Bob",     "Marketing",   72000, 3),
    ("Charlie", "Engineering", 110000, 8),
    ("Diana",   "HR",          65000, 2),
    ("Eve",     "Marketing",   85000, 6),
    ("Frank",   "Engineering", 98000, 7),
]
columns = ["name", "department", "salary", "years"]
df = spark.createDataFrame(data, columns)
df.show()
print("Schema:", df.dtypes)

## 1. select() — Choose Columns

Like SQL `SELECT`. You can select by name, by `col()`, or with expressions.

In [ ]:
# Select by column name (string)
df.select("name", "salary").show()

# Select using col() — allows expressions
df.select(col("name"), col("salary"), col("salary") * 1.1).show()

# Rename inline with alias()
df.select(
    col("name").alias("employee"),
    col("salary").alias("annual_salary"),
    (col("salary") * 0.10).alias("bonus")
).show()

## 2. filter() and where() — Filter Rows

`filter()` and `where()` are **identical** — use whichever reads more naturally.  
Combine conditions with `&` (AND), `|` (OR), `~` (NOT).

In [ ]:
# Single condition
df.filter(col("salary") > 80000).show()

# where() — identical to filter()
df.where(col("department") == "Engineering").show()

# AND condition — must use & with parentheses
df.filter((col("salary") > 70000) & (col("years") >= 5)).show()

# OR condition
df.filter((col("department") == "Engineering") | (col("department") == "Marketing")).show()

# String filter shorthand
df.filter("salary > 90000").show()   # SQL string syntax also works

## 3. withColumn() — Add or Modify a Column

`withColumn(name, expression)` — if `name` already exists, it **overwrites** it; otherwise adds a new column.

In [ ]:
# Add new columns
df2 = df.withColumn("bonus",      col("salary") * 0.10)
df2 = df2.withColumn("dept_upper", upper(col("department")))
df2 = df2.withColumn("salary_k",   spark_round(col("salary") / 1000, 1))
df2 = df2.withColumn("is_senior",  col("years") >= 5)   # boolean column
df2.show()

# Overwrite an existing column (apply a 10% raise)
df3 = df.withColumn("salary", col("salary") * 1.10)
df3.show()

## 4. drop() — Remove Columns

In [ ]:
# Drop a single column
df.drop("years").show()

# Drop multiple columns
df.drop("years", "department").show()

# withColumnRenamed — rename without rewriting everything
df.withColumnRenamed("name", "employee_name") \
  .withColumnRenamed("salary", "annual_salary").show()

## 5. orderBy() / sort() — Sort Rows

Both are identical. Default is ascending. Use `.desc()` for descending.

In [ ]:
# Ascending (default)
df.orderBy("salary").show()

# Descending
df.orderBy(col("salary").desc()).show()

# Sort by multiple columns
df.sort("department", col("salary").desc()).show()

## 6. distinct() and limit()

In [ ]:
# Unique values in a column
df.select("department").distinct().show()

# Unique rows across all columns
df.distinct().count()

# limit() — first N rows (useful for sampling large datasets)
df.limit(3).show()

## 7. Method Chaining — Real-World ETL Pipeline

Transformations can be chained — each returns a new DataFrame.

In [ ]:
# Build an Engineering salary report — chain everything
report = (
    df
    .filter(col("department") == "Engineering")
    .withColumn("bonus",       col("salary") * 0.15)
    .withColumn("total_comp",  col("salary") + col("bonus"))
    .select(
        col("name").alias("employee"),
        col("salary"),
        col("bonus"),
        col("total_comp")
    )
    .orderBy(col("total_comp").desc())
)
report.show()
print("Total Engineering payroll: $", report.agg({"total_comp": "sum"}).collect()[0][0])

## Quick Reference

```python
from pyspark.sql.functions import col, upper, lower, lit, round

df.select("col1", "col2")                        # choose columns
df.select(col("salary") * 1.1)                   # with expression
df.filter(col("salary") > 50000)                 # filter rows
df.where(col("dept") == "Engineering")           # identical to filter
df.filter((col("a") > 1) & (col("b") < 10))     # AND
df.filter((col("a") > 1) | (col("b") < 10))     # OR
df.withColumn("bonus", col("salary") * 0.1)      # add/overwrite column
df.drop("col1", "col2")                          # remove columns
df.withColumnRenamed("old", "new")               # rename
df.orderBy(col("salary").desc())                 # sort descending
df.distinct()                                    # remove duplicate rows
df.limit(10)                                     # first 10 rows
```

## Interview Questions

1. What is the difference between a transformation and an action in PySpark?
2. Are `filter()` and `where()` different? When would you use each?
3. What does `withColumn()` do if the column name already exists?
4. How do you apply AND / OR conditions in `filter()`? Why must conditions be in parentheses?
5. What is lazy evaluation and why does PySpark use it?
6. What is the difference between `select()` and `withColumn()`?
7. How would you rename multiple columns efficiently?

## Practice Exercises

**Easy:**
1. Select only `name` and `department` columns from `df`.
2. Filter employees with more than 4 years of experience.
3. Add a column `tax` that is 30% of `salary`.

**Medium:**
4. Filter employees in Engineering OR Marketing with salary above 80,000.
5. Add a `grade` column: `"Senior"` if years >= 5, else `"Junior"` (use `when/otherwise`).
6. Drop the `years` column and rename `salary` to `annual_pay`.

**Advanced:**
7. Build a complete pipeline: filter dept = HR or Marketing, add 15% bonus, select name/salary/bonus, sort by bonus desc.
8. Find the top 3 earners across all departments.

In [ ]:
spark.stop()